# Colab GPU – fortsetzbarer Zoomania-Test
**Manuell eine Colab-GPU-Sitzung starten.** Danach arbeitet die Zelle selbständig weiter, solange die Laufzeit verfügbar bleibt. Bei Unterbrechung bleiben fertige 10-Sekunden-Segmente in Drive gespeichert. Erneutes Ausführen setzt fort. GitHub kann kostenlose Colab-Sitzungen nicht künstlich offenhalten oder neu starten.


In [ ]:
# Unterbrechungsfester GPU-Test: 60 Sekunden, 6 x 10 Sekunden.
# Drive speichert fertige Segmente. Ein Neustart überspringt vorhandene Segmente.
!pip -q install google-api-python-client
from google.colab import auth
auth.authenticate_user()
import google.auth, subprocess, json, os, time
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive=build('drive','v3',credentials=creds,cache_discovery=False)
source_id='1_fD9iNP8uDkV81L1ne-03sTEa6ERj1Ol'
job='Zoomania_4K_GPU_Test'
work=Path('/content/gpu_convert'); work.mkdir(exist_ok=True)
source=work/'input.mp4'
def find_folder(name,parent=None):
    q="name = '"+name.replace("'","\\'")+"' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    if parent: q+=" and '"+parent+"' in parents"
    r=drive.files().list(q=q,fields='files(id,name)',page_size=100).execute()['files']
    if r: return r[0]['id']
    body={'name':name,'mimeType':'application/vnd.google-apps.folder'}
    if parent: body['parents']=[parent]
    return drive.files().create(body=body,fields='id').execute()['id']
def find_file(name,parent):
    q="name = '"+name.replace("'","\\'")+"' and '"+parent+"' in parents and trashed = false"
    r=drive.files().list(q=q,fields='files(id,name,size)',page_size=100).execute()['files']
    return next((f for f in r if int(f.get('size','0'))>0),None)
def download(file_id,path):
    request=drive.files().get_media(fileId=file_id)
    with open(path,'wb') as f:
        dl=MediaIoBaseDownload(f,request,chunksize=16*1024*1024)
        done=False
        while not done:
            status,done=dl.next_chunk()
            if status: print('Download',round(status.progress()*100,1),'%',end='\r')
    print()
def upload(path,parent,name):
    media=MediaFileUpload(str(path),mimetype='video/mp4',resumable=True,chunksize=8*1024*1024)
    req=drive.files().create(body={'name':name,'parents':[parent]},media_body=media,fields='id,name,webViewLink')
    result=None
    while result is None:
        status,result=req.next_chunk()
        if status: print('Upload',name,round(status.progress()*100,1),'%',end='\r')
    print()
    return result
def run(args):
    subprocess.run(args,check=True)
print('Prüfe NVIDIA-GPU und Encoder...')
run(['nvidia-smi'])
enc=subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True,check=True).stdout
if 'h264_nvenc' not in enc: raise RuntimeError('NVENC fehlt. Der Test darf nicht still auf CPU umschalten.')
root=find_folder('Video Converter')
target=find_folder('Fertig',root)
parts_folder=find_folder(job+'_Segmente',target)
final_name=job+'.mp4'
finished=find_file(final_name,target)
if finished:
    print('Schon fertig:',finished['id'])
else:
    # Segmentdateien sind persistente Checkpoints, nicht in GitHub gespeichert.
    for index in range(6):
        name=f'{job}_part_{index:02d}.mp4'
        if find_file(name,parts_folder):
            print('Vorhandenen Abschnitt überspringen:',name)
            continue
        if not source.exists():
            print('Lade Quelldatei einmalig herunter (11,2 GB; genug Colab-Speicher erforderlich)...')
            download(source_id,source)
        part=work/name
        print('GPU-Konvertierung Abschnitt',index+1,'von 6')
        run(['ffmpeg','-hide_banner','-y','-ss',str(index*10),'-i',str(source),'-t','10',
             '-map','0:v:0','-map','0:a:0?','-vf','scale=3840:2160:flags=lanczos,format=yuv420p',
             '-c:v','h264_nvenc','-b:v','20M','-maxrate','20M','-bufsize','40M',
             '-c:a','aac','-b:a','384k','-ac','6','-movflags','+faststart',str(part)])
        upload(part,parts_folder,name)
        part.unlink(missing_ok=True)
    source.unlink(missing_ok=True)
    # Für den Abschluss alle Segmente aus Drive laden und ohne erneute Kodierung zusammenfügen.
    listfile=work/'concat.txt'
    lines=[]
    for index in range(6):
        name=f'{job}_part_{index:02d}.mp4'
        item=find_file(name,parts_folder)
        if not item: raise RuntimeError('Abschnitt fehlt: '+name)
        path=work/name
        download(item['id'],path)
        lines.append("file '"+str(path)+"'")
    listfile.write_text('\n'.join(lines)+'\n')
    output=work/final_name
    run(['ffmpeg','-hide_banner','-y','-f','concat','-safe','0','-i',str(listfile),'-c','copy',str(output)])
    result=upload(output,target,final_name)
    print('FERTIG:',result)
    for p in work.glob('*.mp4'): p.unlink(missing_ok=True)
